# Feature Analysis

This notebook provides a quantitative analysis of the performance features engineered in the previous step. Through comparative bar charts and delta plots, it identifies distinct differences between teammates regarding speed, braking behavior, throttle application, and gear management across each corner. Additionally, it explores correlations between different performance metrics to uncover the relationship between driving style and speed.

## Setup and Dependencies

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [ ]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.config import *
from src.load_save_data import *
from src.preprocess_data import *
from src.explore_data import *
from src.feature_engineering import *

## Load Processed Data

Fetch cleaned interpolated telemetry data and newly added features for both teammates. 

In [ ]:
session = fastf1.get_session(YEAR, GRAND_PRIX, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, TEAM)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(YEAR, GRAND_PRIX, segment, driver_1, driver_2)
except Exception as e:
    print(f"Processed data for {YEAR} {GRAND_PRIX} {driver_1} and {driver_2} not found: {e}")

## Segment Times Analysis

In [ ]:
fig = barplot_feature_comparison(lap_1, lap_2, "SegmentTime", y_unit="s")
save_figure(fig, "features", "comparison", "SegmentTime", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)
fig = barplot_feature_delta(lap_1, lap_2, "SegmentTime", y_unit="s")
save_figure(fig, "features", "delta", "SegmentTime", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)

## Segments' Speed Features Analysis

Evaluate performance by comparing entry, minimum, and exit speeds for each turn. These visualizations highlight differences in cornering technique, identifying where one driver carries more speed or optimizes their line compared to their teammate.

In [ ]:
for feature in ["MeanSpeed", "StdSpeed", "MaximumSpeed", "MinimumSpeed"]:
    fig = barplot_feature_comparison(lap_1, lap_2, feature, "m/s")
    save_figure(fig, "features", "comparison", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)
    fig = barplot_feature_delta(lap_1, lap_2, feature, "m/s")
    save_figure(fig, "features", "delta", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)

## Segments' Throttle Features  Analysis

Compare mean and standard deviation of throttle usage, as well as the percentage of the segment spent at full throttle. These charts visualize traction management and power deployment differences upon corner exit.

In [ ]:
for feature, y_unit in [("MeanThrottle", ""), ("StdThrottle", ""), ("FullThrottlePercentage", "%")]:
    fig = barplot_feature_comparison(lap_1, lap_2, feature, y_unit=y_unit)
    save_figure(fig, "features", "comparison", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)
    fig = barplot_feature_delta(lap_1, lap_2, feature, y_unit=y_unit)
    save_figure(fig, "features", "delta", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)

## Segments' Gear Shift Features Analysis

Examine transmission-related data, including the number of gear shifts, gear range (highest/lowest gear used), and engine RPM statistics. This identifies variations in gear selection strategies and how each driver manages power through different sections of the track.

In [ ]:
for feature, y_unit in [("NumberOfShifts", ""), ("LowestGear", ""), ("HighestGear", ""), ("MeanRPM", "RPM"), ("StdRPM", "RPM")]:
    fig = barplot_feature_comparison(lap_1, lap_2, feature, y_unit=y_unit)
    save_figure(fig, "features", "comparison", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)
    fig = barplot_feature_delta(lap_1, lap_2, feature, y_unit=y_unit)
    save_figure(fig, "features", "delta", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)

## Turns' Speed Features Analysis

In [ ]:
for feature in ["EntrySpeed", "ApexSpeed", "ExitSpeed"]:
    fig = barplot_feature_comparison(lap_1, lap_2, feature, y_unit="m/s")
    save_figure(fig, "features", "comparison", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)
    fig = barplot_feature_delta(lap_1, lap_2, feature, y_unit="m/s")
    save_figure(fig, "features", "delta", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)

## Turns' Braking Features Analysis

Analyze differences in braking points and duration. This section helps identify which driver commits more aggressively to corner entry and how braking behavior impacts overall lap time.

In [ ]:
for feature in ["BrakingPoint", "BrakingDuration", "BrakingDistance"]:
    fig = barplot_feature_comparison(lap_1, lap_2, feature, y_unit="m" if feature == "BrakingPoint" else "s")
    save_figure(fig, "features", "comparison", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)
    fig = barplot_feature_delta(lap_1, lap_2, feature, y_unit="m" if feature == "BrakingPoint" else "s")
    save_figure(fig, "features", "delta", f"{feature.lower()}", YEAR, GRAND_PRIX, TEAM, driver_1, driver_2)

## Features and Time Correlation Analysis

In [ ]:
time_correlations = []

time_relationships = [
    # Segment Features -> Segment Time
    ("ExitSpeed", "SegmentTime"),
    ("MinimumSpeed", "SegmentTime"),
    ("MeanSpeed", "SegmentTime"),
    ("StdSpeed", "SegmentTime"),
    ("BrakingPoint", "SegmentTime"),
    ("BrakingDistance", "SegmentTime"),
    ("MeanThrottle", "SegmentTime"),
    ("StdThrottle", "SegmentTime"),
    ("FullThrottlePercentage", "SegmentTime"),
    ("NumberOfShifts", "SegmentTime"),
    ("MinimumRPM", "SegmentTime"),
    ("MaximumRPM", "SegmentTime"),
]

for x_feature, y_feature in time_relationships:

    correlation = calculate_feature_correlation(
        lap_1,
        lap_2,
        x_feature,
        y_feature,
    )

    time_correlations.append(correlation)

time_correlations = (
    pd.concat(time_correlations, ignore_index=True)
    .sort_values("Pearson", key=lambda s: s.abs(), ascending=False)
    .reset_index(drop=True)
)

display(time_correlations)

plots = [
    ("MeanSpeed", "SegmentTime"),
    ("MinimumSpeed", "SegmentTime"),
    ("MeanThrottle", "SegmentTime"),
    ("FullThrottlePercentage", "SegmentTime"),
    ("BrakingPoint", "SegmentTime"),
]

for x_feature, y_feature in plots:
    fig = scatterplot_features_relationship(lap_1, lap_2, x_feature, y_feature)
    save_figure(
        fig,
        "features",
        "relationship",
        f"{x_feature.lower()}_{y_feature.lower()}",
        YEAR,
        GRAND_PRIX,
        TEAM,
        driver_1,
        driver_2,
    )

## Feature Relationship Correlation Analysis

Investigate relationships between different performance metrics (e.g., Entry Speed vs. Minimum Speed or Braking Point vs. Minimum Speed). Scatter plots here reveal how specific driving styles like early braking or throttle modulation correlate with exit speed and overall corner performance.

In [ ]:
feature_correlations = []

feature_relationships = [
    # Same corner
    ("BrakingPoint", "EntrySpeed"),
    ("BrakingPoint", "MinimumSpeed"),
    ("BrakingDistance", "EntrySpeed"),
    ("BrakingDistance", "MinimumSpeed"),
    ("EntrySpeed", "MinimumSpeed"),

    # Same segment
    ("MeanThrottle", "MeanSpeed"),
    ("FullThrottlePercentage", "MeanSpeed"),

    # Corner transition
    ("ExitSpeed", "EntrySpeed"),
    ("ExitSpeed", "MeanSpeed"),

    # Overall speed behaviour
    ("MeanSpeed", "MinimumSpeed"),
    ("MeanSpeed", "EntrySpeed"),

    # Consistency
    ("StdSpeed", "MinimumSpeed"),
]

for x_feature, y_feature in feature_relationships:

    correlation = calculate_feature_correlation(
        lap_1,
        lap_2,
        x_feature,
        y_feature,
    )

    feature_correlations.append(correlation)
    
feature_correlations = (
    pd.concat(feature_correlations, ignore_index=True)
    .sort_values("Pearson", key=lambda s: s.abs(), ascending=False)
    .reset_index(drop=True)
)

display(feature_correlations)

plots = [
    ("MeanSpeed", "MinimumSpeed"),
    ("MeanThrottle", "MeanSpeed"),
    ("EntrySpeed", "MinimumSpeed"),
    ("BrakingPoint", "MinimumSpeed"),
    ("ExitSpeed", "EntrySpeed"),  # para investigar

]

for x_feature, y_feature in plots:
    fig = scatterplot_features_relationship(lap_1, lap_2, x_feature, y_feature)
    save_figure(
        fig,
        "features",
        "relationship",
        f"{x_feature.lower()}_{y_feature.lower()}",
        YEAR,
        GRAND_PRIX,
        TEAM,
        driver_1,
        driver_2,
    )